# BRACS

In [7]:
from pathlib import Path

ROOT_DIR = Path( "/mnt/data/Public/Public/breast/bracs/histoimage.na.icar.cnr.it" )
ANNOT_DIR = ROOT_DIR / "BRACS_WSI_Annotations"
SLIDE_DIR = ROOT_DIR / "BRACS_WSI"

In [8]:
import mlflow
import os

os.environ["MLFLOW_TRACKING_USERNAME"] = "..."
os.environ["MLFLOW_TRACKING_PASSWORD"] = "..."
mlflow.set_tracking_uri("https://mlflow.rationai.cloud.e-infra.cz/")

## Metadata

Table containing all the slide-level info

In [9]:
import pandas as pd

df = pd.read_excel(str(ROOT_DIR / "BRACS.xlsx"))

In [10]:
df.columns

Index(['WSI Filename', 'Patient Id', 'RoI ', 'WSI label', 'Set'], dtype='object')

Slide-Level Grades

In [11]:
set( df["WSI label"].values )

{'ADH', 'DCIS', 'FEA', 'IC', 'N', 'PB', 'UDH'}

In [12]:
df.head()

,WSI Filename,Patient Id,RoI,WSI label,Set
0,BRACS_264,85,24,PB,Testing
1,BRACS_265,87,18,UDH,Validation
2,BRACS_280,30,19,IC,Training
3,BRACS_281,111,37,IC,Training
4,BRACS_283,109,49,IC,Training


### Sumarize Counts

In [13]:
wsi_label_categories = {
    "Negative": ["N", "PB", "UDH"],
    "Atypical": ["FEA", "ADH"],
    "Positive": ["DCIS", "IC"],
}
short_names = {"Negative": "neg.", "Atypical": "atyp.", "Positive": "pos."}
split_order = ["Training", "Validation", "Testing"]

In [14]:
!uv pip install --python ~/carcinoma-binary-classification-methods/.venv/bin/python tabulate

Using Python 3.12.3 environment at: /home/jovyan/carcinoma-binary-classification-methods/.venv
Audited 1 package in 232ms


In [15]:
def fmt(total, parts) -> str:
    return f"{total} ({' / '.join(map(str, parts))})"

def summarize(df: pd.DataFrame) -> pd.Series:
    row = {"# patients": df["Patient Id"].nunique()}
    label_counts = df["WSI label"].value_counts()
    group_totals = []
    for cat, labels in wsi_label_categories.items():
        counts = label_counts.reindex(labels, fill_value=0)
        col = f"# {short_names[cat]} slides ({' / '.join(labels)})"
        row[col] = fmt(counts.sum(), counts)
        group_totals.append(counts.sum())

    group_col = f"# slides ({' / '.join(short_names.values())})"
    row[group_col] = fmt(sum(group_totals), group_totals)
    return pd.Series(row)


table = pd.DataFrame({split: summarize(g) for split, g in df.groupby("Set")}).T
table = table.loc[split_order]
table.loc["Total"] = summarize(df)

print(table.to_markdown())

|            |   # patients | # neg. slides (N / PB / UDH)   | # atyp. slides (FEA / ADH)   | # pos. slides (DCIS / IC)   | # slides (neg. / atyp. / pos.)   |
|:-----------|-------------:|:-------------------------------|:-----------------------------|:----------------------------|:---------------------------------|
| Training   |          133 | 203 (27 / 120 / 56)            | 52 (24 / 28)                 | 140 (40 / 100)              | 395 (203 / 52 / 140)             |
| Validation |           26 | 30 (10 / 11 / 9)               | 14 (6 / 8)                   | 21 (9 / 12)                 | 65 (30 / 14 / 21)                |
| Testing    |           31 | 32 (7 / 16 / 9)                | 23 (11 / 12)                 | 32 (12 / 20)                | 87 (32 / 23 / 32)                |
| Total      |          189 | 265 (44 / 147 / 74)            | 89 (41 / 48)                 | 193 (61 / 132)              | 547 (265 / 89 / 193)             |


### Export Enhanced Metadata Table

In [16]:
all_slides = list( SLIDE_DIR.rglob("*.svs") )
len(all_slides)

547

Attach slide path to the table

In [17]:
stem_to_path = { p.stem: p for p in all_slides }
wsi_label_to_category = {
    "N" : "Negative",
    "PB" : "Negative",
    "UDH" : "Negative",
    "FEA": "Atypical",
    "ADH": "Atypical",
    "DCIS": "Positive",
    "IC": "Positive",
}

In [18]:
df["slide_path"] = df["WSI Filename"].map( stem_to_path )
df["Category"] = df["WSI label"].map( wsi_label_to_category )

In [19]:
df.head()

,WSI Filename,Patient Id,RoI,WSI label,Set,slide_path,Category
0,BRACS_264,85,24,PB,Testing,/mnt/data/Public/Public/breast/bracs/histoimag...,Negative
1,BRACS_265,87,18,UDH,Validation,/mnt/data/Public/Public/breast/bracs/histoimag...,Negative
2,BRACS_280,30,19,IC,Training,/mnt/data/Public/Public/breast/bracs/histoimag...,Positive
3,BRACS_281,111,37,IC,Training,/mnt/data/Public/Public/breast/bracs/histoimag...,Positive
4,BRACS_283,109,49,IC,Training,/mnt/data/Public/Public/breast/bracs/histoimag...,Positive


In [20]:
df.to_csv("./bracs_full.csv", index=False)
mlflow.set_experiment("Breast Cancer")

with mlflow.start_run(run_name="BRACS Metadata"):
    mlflow.log_artifact("./bracs_full.csv")

🏃 View run BRACS Metadata at: https://mlflow.rationai.cloud.e-infra.cz/#/experiments/170/runs/f4217d25db964834a2527412b8758f90
🧪 View experiment at: https://mlflow.rationai.cloud.e-infra.cz/#/experiments/170


## Annotations

In [21]:
from collections import Counter
from pathlib import Path
from xml.etree import ElementTree as ET
import json

def scan_annotation_groups(annotation_dir: str | Path) -> Counter[str]:
    annotation_dir = Path(annotation_dir)
    group_counts: Counter[str] = Counter()

    for path in annotation_dir.rglob("*.xml"):
        root = ET.parse(path).getroot()
        for region in root.findall(".//Annotation"):
            group = region.get("PartOfGroup")
            if group is not None:
                group_counts[group] += 1

    for path in annotation_dir.rglob("*.geojson"):
        with open(path) as f:
            data = json.load(f)
        features = data if isinstance(data, list) else data["features"]
        for feature in features:
            classification = feature.get("properties", {}).get("classification", {})
            name = classification.get("name")
            if name is not None:
                group_counts[name] += 1
            else: group_counts["<UNCLASSIFIED>"] += 1

    return group_counts
    
counts = scan_annotation_groups(ANNOT_DIR)

In [22]:
counts

Counter({'Pathological-benign': 822,
         'DCIS-sure': 789,
         'FEA-sure': 755,
         'Malignant-sure': 593,
         'UDH-sure': 511,
         'ADH-sure': 503,
         'Benign-sure': 477,
         'MALIGNANT': 38,
         'Malignant': 18,
         'Pathologica benign': 14,
         'Benign sure': 7,
         'UDH': 6,
         'ADH': 3,
         'BENIGN': 1,
         'Pathological-benign (Benign-sure)': 1,
         'FEA': 1,
         'DCIS': 1})

In [23]:
from collections import Counter

mapping = {
    "benign_pathological": [
        "Pathological-benign",
        "Pathologica benign",
        "Pathological-benign (Benign-sure)",
    ],
    "benign": [
        "Benign-sure",
        "Benign sure",
        "BENIGN",
    ],
    "dcis": [
        "DCIS-sure",
        "DCIS",
    ],
    "fea": [
        "FEA-sure",
        "FEA",
    ],
    "malignant": [
        "Malignant-sure",
        "MALIGNANT",
        "Malignant",
    ],
    "udh": [
        "UDH-sure",
        "UDH",
    ],
    "adh": [
        "ADH-sure",
        "ADH",
    ],
}

def normalize_counts(raw_counts: Counter, mapping: dict[str, list[str]]) -> Counter:
    # build reverse lookup: raw label -> canonical key
    reverse = {
        raw_label: canonical
        for canonical, raw_labels in mapping.items()
        for raw_label in raw_labels
    }

    normalized: Counter = Counter()
    unmapped: Counter = Counter()

    for raw_label, count in raw_counts.items():
        canonical = reverse.get(raw_label)
        if canonical is not None:
            normalized[canonical] += count
        else:
            unmapped[raw_label] += count

    if unmapped:
        print(f"WARNING: {sum(unmapped.values())} annotations had unmapped labels: {dict(unmapped)}")

    return normalized


normalized = normalize_counts(counts, mapping)
print(normalized)

Counter({'benign_pathological': 837, 'dcis': 790, 'fea': 756, 'malignant': 649, 'udh': 517, 'adh': 506, 'benign': 485})


These were created after the initial exploration:

In [24]:
annot_masks = Path( mlflow.artifacts.download_artifacts("mlflow-artifacts:/170/e4324d7f10634a40a319cbfd88f93d88/artifacts/annotation_masks") )

## Slides

In [25]:
import openslide

mpps = set()
level_counts = set()
downsamples = set()

for path in df["slide_path"]:
    slide = openslide.OpenSlide(path)
    mpp_x = slide.properties.get(openslide.PROPERTY_NAME_MPP_X)
    mpp_y = slide.properties.get(openslide.PROPERTY_NAME_MPP_Y)

    mpps.add( (mpp_x, mpp_y) )
    level_counts.add( slide.level_count )
    downsamples.add( slide.level_downsamples )

    slide.close()

In [26]:
mpps

{('0.25190000000000001', '0.25190000000000001'),
 ('0.25240000000000001', '0.25240000000000001')}

In [27]:
level_counts

{3, 4}

In [28]:
downsamples

{(1.0, 4.0, 16.0, 32.0),
 (1.0, 4.0, 16.0, 32.002073075926404),
 (1.0, 4.0, 16.0, 32.0025706940874),
 (1.0, 4.0, 16.0, 32.00279427174293),
 (1.0, 4.0, 16.0, 32.00290803344238),
 (1.0, 4.0, 16.0, 32.002952029520294),
 (1.0, 4.0, 16.0, 32.00306044376435),
 (1.0, 4.0, 16.0, 32.00319233838787),
 (1.0, 4.0, 16.0, 32.00324807145758),
 (1.0, 4.0, 16.0, 32.00338266384778),
 (1.0, 4.0, 16.0, 32.00344976282881),
 (1.0, 4.0, 16.0, 32.00352422907489),
 (1.0, 4.0, 16.0, 32.00494437577256),
 (1.0, 4.0, 16.0, 32.00568027462893),
 (1.0, 4.0, 16.0, 32.00600343238081),
 (1.0, 4.0, 16.0, 32.00680878804478),
 (1.0, 4.0, 16.0, 32.00761365947294),
 (1.0, 4.0, 16.0, 32.008481452885384),
 (1.0, 4.0, 16.0003610760065, 32.003611412062114),
 (1.0, 4.0, 16.0003612716763, 32.002795619279006),
 (1.0, 4.0, 16.000361402240692, 64.01691002371965),
 (1.0, 4.0, 16.000361467558285, 64.01479209466285),
 (1.0, 4.0, 16.00039009167154, 32.003901677721416),
 (1.0, 4.0, 16.000390167772142, 32.00357460728721),
 (1.0, 4.0, 16.00

## Local Visualization

In [29]:
!uv pip install --python ~/carcinoma-binary-classification-methods/.venv/bin/python xopat

Using Python 3.12.3 environment at: /home/jovyan/carcinoma-binary-classification-methods/.venv
Audited 1 package in 31ms


In [30]:
def get_masks(slide: pd.Series) -> list[tuple[str, str]]:
    stem = slide["WSI Filename"]
    print(stem)
    occurences = list( annot_masks.rglob(f"*{stem}*") )
    res = []
    for occ in occurences:
        parts = str(occ).split("/")
        res.append( (parts[-2], str(occ)) )

    return res

In [31]:
import xopat

xopat.setup_jupyterhub("https://hub.rationai.cloud.e-infra.cz")

slide = df.iloc[380]

data_dir = "/"
slide_name = str( slide["slide_path"] )
masks = get_masks(slide)

server = xopat.run_server(data_dir=data_dir)

session_config = {

    "data": [slide_name] + [ path for name, path in masks ],
    "background": [{"dataReference": 0}],
    "visualizations": [{
            "name": name,
            "shaders": {
                "heatmap": {
                    "type": "heatmap",
                    "dataReferences": [i + 1],
                    "params": {
                        "color": "#FF0000",
                        "opacity": 0.7
                    }
                }
            }
        } for i, (name, path) in enumerate(masks) ]
}

xopat.display(server, session_config)

Configured for JupyterHub: https://hub.rationai.cloud.e-infra.cz/user/adam-dzadon/proxy/9001/
BRACS_1932
WSI-Service: using cached binary /home/jovyan/.xopat/wsi/wsi-v1.0.2/linux/wsi_service_binary (wsi-v1.0.2)
xOpat: using cached binary /home/jovyan/.xopat/xopat/xopat-v1.0.7/linux/xopat_binary (xopat-v1.0.7)
Starting WSI-Service...
WSI-Service is running.
Starting xOpat...
xOpat is running.
Servers running. Slides folder: /
